# Temporal Conflict Resolution — Reconciliation Dashboard

Runs the engine against `sample_inputs/comprehensive_all_cases.json` and visualizes:
- final status distribution across transactions
- conflicts detected per transaction
- version count (how many times each transaction's state changed) per transaction

Requires: `pandas`, `matplotlib` (`pip install -r requirements.txt`).

In [ ]:
import os, sys, json

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models import TransactionEvent
from src.database import Database
from src.reconciliation_engine import ReconciliationEngine

input_path = os.path.join(PROJECT_ROOT, "sample_inputs", "comprehensive_all_cases.json")
schema_path = os.path.join(PROJECT_ROOT, "schema.sql")

with open(input_path) as f:
    raw_events = json.load(f)

events = [TransactionEvent(**e) for e in raw_events]

db = Database(":memory:", schema_path=schema_path)
engine = ReconciliationEngine(db)
results = engine.process_events(events)
report = engine.generate_final_report()

print(f"Processed {len(events)} events across {len(report)} transactions.")

In [ ]:
import pandas as pd

rows = []
for entry in report:
    fs = entry["final_state"]
    rows.append({
        "transaction_id": entry["transaction_id"],
        "version": fs["version"],
        "source": fs["source"],
        "amount": fs["amount"],
        "currency": fs["currency"],
        "status": fs["status"],
        "events_considered": entry["events_considered"],
        "conflicts_detected": entry["conflicts_detected"],
    })

df = pd.DataFrame(rows).set_index("transaction_id")
df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

df["conflicts_detected"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Conflicts detected per transaction")
axes[0].set_ylabel("conflicts")
axes[0].tick_params(axis="x", rotation=60)

df["version"].plot(kind="bar", ax=axes[1], color="#55A868")
axes[1].set_title("Final version per transaction\n(number of accepted state changes)")
axes[1].set_ylabel("version")
axes[1].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()

In [ ]:
status_counts = df["status"].fillna("(none)").value_counts()

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(status_counts.values, labels=status_counts.index, autopct="%1.0f%%", startangle=90)
ax.set_title("Final status distribution")
plt.show()

## Audit trail drill-down

Pick any `transaction_id` from the table above to see its full, ordered audit trail —
every event considered, every conflict detected, and every resolution applied.

In [ ]:
def show_audit_trail(transaction_id):
    trail = db.get_audit_trail(transaction_id)
    print(f"Audit trail for {transaction_id}:\n")
    for entry in trail:
        print(f"  [{entry['created_at']}] {entry['action']}")
        for k, v in entry["details"].items():
            print(f"      {k}: {v}")
    conflicts = db.get_conflicts(transaction_id)
    if conflicts:
        print(f"\n  Conflicts ({len(conflicts)}):")
        for c in conflicts:
            print(f"    [{c['conflict_field']}] {c['existing_source']}({c['existing_value']}) "
                  f"vs {c['incoming_source']}({c['incoming_value']}) "
                  f"-> resolved by {c['resolution_source']}: {c['resolution_reason']}")

# Example: the invalid-status-transition case
show_audit_trail("TXN-S5-001")

In [ ]:
# Example: the late-event / out-of-order case
show_audit_trail("TXN-S3-001")

In [ ]:
db.close()